# Administration

* Find the location of configuration files
    * `SELECT name, setting FROM pg_settings WHERE category = 'File Locations';`
 
* If the context is postmaster, you’ll need a restart. If the context is user, a reload will suffice.

* **postgresql.conf** Controls general settings
* **pg_hba.conf** Controls access to the server, dictating which users can log in to which databases, which IP addresses can connect, and which authentication scheme to accept.

```
select context, pending_restart, name, setting, category, short_desc from pg_settings 
where category!='File Locations' and name='port'
order by context;
```

* A reload can be done in several ways
    *  `pg_ctl reload -D your_data_directory_here`
    *  `service postgresql-9.5 reload`
    *  `SELECT pg_reload_conf();`
 
* Instead of editing postgresql.conf directly, you should override settings using an additional file called postgresql.auto.conf

## Settings Context
* User settings can be changed by each user to affect just that user’s sessions.
    * If set by the superuser, the setting becomes a default for all users who connect after a reload. 
* Postmaster settings affect the entire server (postmaster represents the PostgreSQL service) and take effect only after a restart.
* Settings with user or superuser context can be set for a specific database, user, session, and function level

Display current setting in intuitive units
* `SHOW shared_buffers;`

```
SELECT name, sourcefile, sourceline, setting, applied
FROM pg_file_settings
WHERE name IN ('listen_addresses','deadlock_timeout','shared_buffers',
    'effective_cache_size','work_mem','maintenance_work_mem')
ORDER BY name;
```

**pg_file_settings** shows where a particular setting can be found in the source file.  In cases where a particular setting is present in both postgresql.conf and postgresql.auto.conf, the postgresql.auto.conf one will take precedent. The applied setting tells if the setting is in effect.

## Create Login Roles
* `CREATE ROLE leo LOGIN PASSWORD 'king' VALID UNTIL 'infinity' CREATEDB;`
* `CREATE ROLE regina LOGIN PASSWORD 'queen' VALID UNTIL '2020-1-1 00:00' SUPERUSER;`
    * Role with superuser privileges
 
## Create Group Roles
* `CREATE ROLE royalty INHERIT;`
    * Any member of royalty will automatically inherit privileges of the royalty role, except for the superuser privilege
    * INHERIT is the default
* `CREATE ROLE staff NONINHERIT;`
    * To refrain from passing privileges from the group to its members
* `SET ROLE royalty;`
    * Does not require superuser rights
    * Users can impersonate their group role this way
    * `SELECT current_user`
      * Changes the current_user
* `SET SESSION AUTHORIZATION`
    * Available to people who log in as superusers
    * `SELECT session_user, current_user`
        * Changes both session_user, current_user
    * In effect for the life of the session

## Template Databases

* The basic syntax to create a database modeled after a specific template is:
    * `CREATE DATABASE my_db TEMPLATE my_template_db;`
        * You can pick any database to serve as the template
        * This will copy all of the data and tables
* You can mark any datbase as a template but then it is not editable. Can set it to false to make edits.
    * `UPDATE pg_database SET datistemplate = TRUE WHERE datname = 'mydb';` 

## Extensions

* `CREATE SCHEMA my_extensions;`
* `ALTER DATABASE mydb SET search_path='$user', public, my_extensions;`
    * When you install extensions, be sure to indicate your new extension schema as their new home.
    * Will take effect after reconnecting
 
* To see extensions you have already installed in a database:
 

```
SELECT name, default_version, installed_version, left(comment,30) As comment
FROM pg_available_extensions
WHERE installed_version IS NOT NULL
ORDER BY name;
```

* To get details of what is packaged in the extension **fuzzystrmatch**

```
SELECT pg_describe_object(D.classid,D.objid,0) AS description
FROM pg_catalog.pg_depend AS D INNER JOIN pg_catalog.pg_extension AS E
ON D.refobjid = E.oid
WHERE
D.refclassid = 'pg_catalog.pg_extension'::pg_catalog.regclass AND
deptype = 'e' AND
E.extname = 'fuzzystrmatch';
```


* To view all extension binaries already available on your server, enter:
    * `SELECT * FROM pg_available_extensions;` 

## Backup and Restore

* Use **pg_dump** to back up specific databases.
* To back up all databases in plain text along with server globals, use **pg_dumpall**
    * Needs to run under a superuser account so that it back up all databases.
* Use **pg_basebackup** to do system-level disk backup of all databases
* Use **pg_restore** to restore files created using TAR, custom, or directory option of pg_dump

## Disk Storage with Tablespaces

A PostgreSQL cluster automatically begets two tablespaces: 
* **pg_default**, which stores all user data.
* **pg_global**, which stores all system data.

### Create Tablespace
```CREATE TABLESPACE secondary LOCATION 'C:/pgdata94_secondary';```

### Moving Objects among Tablespaces
* Move all objects in the database to your secondary tablespace
    * ```ALTER DATABASE mydb SET TABLESPACE secondary;```
* Move just one table
    * ```ALTER TABLE mytable SET TABLESPACE secondary;```
* To move all objects from default tablespace to secondary tablespace
    * ```ALTER TABLESPACE pg_default MOVE ALL TO secondary;```
 


## Constructs

### ROWS FROM

* This construct allows for the use of multiple set-returning functions in a series, even if they have an unbalanced number of elements in each set:

```
SELECT *
FROM ROWS FROM ( jsonb_each('{"a":"foo1","b":"bar"}'::jsonb),
				 jsonb_each('{"c":"foo2"}'::jsonb) ) 
				 x (a1,a1_val,a2,a2_val);
```

### Muti argument unnest for unbalanced arrays
`SELECT * FROM unnest('{blind,mouse}'::text[], '{1,2,3}'::int[]) AS f(t,i);`

### Array Sliciing and Splicing
`SELECT fact_subcats[1:2] || fact_subcats[3:4] || 4 FROM census.lu_fact_types;`

## Specify Particular JSON Array Index
`SELECT person->'children'->0->'name' FROM persons;`

## Using DO to generate dynamic SQL

```
DO language plpgsql
$$
DECLARE var_sql text;
BEGIN
    var_sql := string_agg(
        $sql$
        INSERT INTO lu_fact_types(category, fact_subcats, short_name)
        SELECT
            'Housing',
            array_agg(s$sql$ || lpad(i::text,2,'0')
              || ') As fact_subcats,'
              || quote_literal('s' || lpad(i::text,2,'0')) || ' As short_name
        FROM staging.factfinder_import
        WHERE s' || lpad(I::text,2,'0') || $sql$ ~ '^[a-zA-Z]+' $sql$, ';'
    )
    FROM generate_series(1,51) As I;
    EXECUTE var_sql;
END
$$;
```

### CTE
* Recursive window functions
* Calling the same window function multiple times by reference

```
SELECT * FROM (
    SELECT
        ROW_NUMBER() OVER( wt ) As rnum, 1
        substring(tract_id,1, 5) As county_code,
        tract_id,
        LAG(tract_id,2) OVER wt As tract_2_before,
        LEAD(tract_id) OVER wt As tract_after
    FROM census.lu_tracts
    WINDOW wt AS (PARTITION BY substring(tract_id,1, 5) ORDER BY tract_id) 2
) As x
WHERE rnum BETWEEN 2 and 3 AND county_code IN ('25007','25025')
ORDER BY county_code, rnum;
```

## Grant

* When granting privileges, you can add WITH GRANT OPTION. This means that the grantee can grant her own privileges to others, passing them on:
    * `GRANT ALL ON ALL TABLES IN SCHEMA public TO mydb_admin WITH GRANT OPTION`
* To grant specific privileges on ALL objects of a specific type use ALL instead of the specific object name, as in:
    * `GRANT SELECT, REFERENCES, TRIGGER ON ALL TABLES IN SCHEMA my_schema TO PUBLIC` 
* To grant privileges to all roles, you can use the PUBLIC alias, as in:
    * `GRANT USAGE ON SCHEMA my_schema TO PUBLIC`
* Able to **REVOKE** privileges as well with REVOKE command

## Default Privileges
`ALTER DEFAULT PRIVILEGES IN SCHEMA my_schema
GRANT ALL ON TABLES TO mydb_admin WITH GRANT OPTION;`   
* GRANT ALL permissions on future tables to role mydb_admin
*  In addition, allow members in mydb_admin to be able to grant a subset or all privileges to other users to future tables in this schema
*  GRANT ALL gives permission to add/update/delete/truncate rows, add triggers, and create constraints on the tables`